# Sequence-to-Seqeunce
* Sequence-to-sequence(Seq2Seq)는 입력된 시퀀스로부터 다른 도메인의 시퀀스를 출력하는 모델
* 대표적인 응용분야
  - **기계번역 (machine translation)**
    - '한국어 도메인' 을 가지는 문장을 입력하면 '영어 도메인' 에 해당하는 문장을 얻을수 있다.
    - 구글 번역기, 파파고...
  - **내용 요약 (Text Summarization)** : 상대적으로 큰 원문의 핵심 내용만 간추려서 상대적으로 작은 요약문으로 변환하는 것
  - **음성 인식 (Speech to Text,STT)** : 음성을 글자로 변환하는 기술

## 인코더 & 디코더

* Seq2Seq는 다른 특별한 기술을 이용하는 것이 아니라, 지금까지 배운 RNN(순환신경망) 기술들을 조합해 만들며, 일반적으로 **encoder**와 **decoder**로 구성
<br>

![](https://static.wikidocs.net/images/page/24996/%EC%8B%9C%ED%80%80%EC%8A%A4%ED%88%AC%EC%8B%9C%ED%80%80%EC%8A%A4.PNG)

![](https://static.wikidocs.net/images/page/24996/seq2seq%EB%AA%A8%EB%8D%B811.PNG)

- Seq2Seq 는 '인코더' 와 '디코더' 라는 모듈로 구성됨.
- **인코더**
  - 입력문장의 모든 단어들을 순차적으로 입력 받은뒤
  - 마지막에 이 모든 단어 정보들을 압축하여 하나의 벡터를 만든다. 이를 **컨텍스트 벡터(context vector)** 라 함
  - 이 컨텍스트 벡터가 디코더에 전송된다.
- **디코더**
  - 컨텍스트 벡터 를 받아서 번역된 단어를 한개씩 순차적으로 출력.



* **Context Vector**

![](https://static.wikidocs.net/images/page/24996/%EC%BB%A8%ED%85%8D%EC%8A%A4%ED%8A%B8_%EB%B2%A1%ED%84%B0.PNG)

위의 그림에서는 컨텍스트 벡터를 4의 사이즈로 표현하였지만,
실제 현업에서 사용되는 seq2seq 모델에서는 보통 수백 이상의 차원을 갖고있습니다

## 인코더와 디코더의 아키텍쳐

![](https://static.wikidocs.net/images/page/24996/%EC%9D%B8%EC%BD%94%EB%8D%94%EB%94%94%EC%BD%94%EB%8D%94%EB%AA%A8%EB%8D%B8.PNG)

- 인코더와 디코더라 불리우는 두개의 RNN 아키텍처로 구성
  - 인코더 : 입력문장을 입력받는 RNN셀
  - 디코더 : 출력문장을 출력하는 RNN셀
  - 일반적으로 바닐라RNN 보다는 **LSTM** 이나 **GRU** 로 구성함 (성능때문에!)

- Seq2Seq 는 **'훈련과정'** 과 **'예측(테스트)과정'** 의 작동방식이 조금 다릅니다.    

### '훈련과정'의 아키텍쳐
- 인코더에는 '**입력 단어**'들이 입력 하여 컨텍스트 벡터 출력
- 디코더에게 인코더가 보낸 '컨텍스트 벡터'와 '**실제 정답**'인 &lt;sos&gt; je suis étudiant를 입력 받았을 때,
je suis étudiant &lt;eos&gt;가 나와야 된다고 **'정답'을 알려주면서 훈련**합니다.

- 즉, 훈련과정에는 위의 **3가지 데이터**가 필요하다!

- 훈련과정에서는 **교사 강요(teacher forcing)** 를 통해 훈련한다  (아래)

## 교사 강요(Teacher Forcing)
* 앞서 설명한 seq2seq 모델을 잘 살펴보면 디코더의 입력이 필요하지 않음을 알 수 있음
* 예측이 잘못됐을 경우, 잘못된 예측이 다음 시점으로 입력돼 연쇄적으로 잘못된 예측을 함
* 이를 해결하기 위해 디코더의 다음 시점의 입력으로 이전 시점의 출력이 아닌, 정답을 주어 이를 방지함
 - "이게 정답이야" 라는 식으로 개입해주는 거다

### '예측(테스트) 과정' 의 아키텍쳐
- 인코더
  - 입력문장은 단어 토큰화 되어 쪼개어 지고
  - 단어토큰 각각은 RNN셀의 각 타임스텝의 입력이 된다.
  - 인코더 RNN셀은 모든 단어를 입력받은 뒤에,
  - 인코더의 마지막 타임스텝 의 은닉상태(hidden state) 를 디코더로 넘겨진다 (이게 바로 context vector 다)
  - 이 context vector 는 디코더 RNN셀의 첫번째 은닉상태에 사용된다.

- 디코더
  - 우선 디코더는 초기 입력으로 문장의 시작을 의미하는 토큰
  &lt;sos&gt; 가 들어감
  - 디코더는 &lt;sos&gt; 가 입력되면 다음에 등장할 확률이 높은 단어를 예측.
  - 위 그림에서 첫번째 타입스텝의 디코더 RNN셀은 다음에 등장할 단어로 "je"를 예측했다.
  - 그렇게 예측한 단어 je 를 다음 타임스텝의 RNN 셀 입력으로 입력
  - .. 다음에는 "suis", 다음에는 "etudiant"..
  - 이런식으로 디코더는 **타임스텝을 거듭 반복**하면서 다음의 단어들을 예측해낸다.  언제까지?
  - 이 반복행위는 문장의 끝을 의미하는 토큰 &lt;eos&gt; 가 다음단어로 예측될때까지 반복됨.

## word embedding
- 아래 그림에서 입,출력에 쓰이는 단어토큰 부분 주목
![](https://static.wikidocs.net/images/page/24996/%EB%8B%A8%EC%96%B4%ED%86%A0%ED%81%B0%EB%93%A4%EC%9D%B4.PNG)

- seq2seq 에 사용되는 모든 단어들은 임베딩 벡터로 변환후 입력으로 사용됨.

하나의 RNN셀은 각각의 timestep 마다 두개의 입력을 받는다

![](https://static.wikidocs.net/images/page/24996/rnn%EA%B7%BC%ED%99%A9.PNG)

## softmax

![](https://static.wikidocs.net/images/page/24996/decodernextwordprediction.PNG)
- 각 타임스텝의 출력 단어로 나올수 있는 단어는 '다양'하다
- 이를 예측하기 위해 softmax 함수 사용됨